## Target analysis of TA of WL-PSI of Scy6803

### Inspect data

In [ ]:
from glotaran.io import load_scheme
from pyglotaran_extras.compat import convert


def _case_study_convert(native_result, scheme):
    """Project native v0.8 results for legacy plotting while retaining native results."""
    import numpy as np
    import xarray as xr

    compat_result = convert(native_result)
    for dataset_label, dataset in compat_result.data.items():
        if "irf_center" in dataset.coords:
            irf_center = dataset.coords["irf_center"]
            if irf_center.ndim and np.allclose(irf_center, irf_center.values.flat[0]):
                dataset = dataset.drop_vars("irf_center").assign_coords(
                    irf_center=float(irf_center.values.flat[0])
                )
                compat_result.data[dataset_label] = dataset
        optimization_result = native_result.optimization_results[dataset_label]
        global_dimension = optimization_result.meta.global_dimension
        model_dimension = optimization_result.meta.model_dimension
        input_data = optimization_result.input_data
        residual = optimization_result.residuals
        if isinstance(input_data, xr.Dataset):
            input_data = input_data["data"]
        if isinstance(residual, xr.Dataset):
            residual = residual["residual"]
        fitted_data = input_data - residual
        if {"time", "spectral"}.issubset(fitted_data.dims):
            fitted_data = fitted_data.transpose("time", "spectral")
        dataset["fitted_data"] = fitted_data
        data_model = next(
            experiment.datasets[dataset_label]
            for experiment in scheme.experiments.values()
            if dataset_label in experiment.datasets
        )
        weight = xr.ones_like(residual)
        for weight_item in data_model.weights:
            selected = xr.ones_like(residual, dtype=bool)
            if weight_item.global_interval is not None:
                lower, upper = weight_item.global_interval
                selected = selected & (
                    (residual.coords[global_dimension] >= lower)
                    & (residual.coords[global_dimension] <= upper)
                )
            if weight_item.model_interval is not None:
                lower, upper = weight_item.model_interval
                selected = selected & (
                    (residual.coords[model_dimension] >= lower)
                    & (residual.coords[model_dimension] <= upper)
                )
            weight = weight * xr.where(selected, float(weight_item.value), 1.0)
        dataset["weight"] = weight
        dataset["weighted_residual"] = residual * weight
        dataset["clp"] = optimization_result.fit_decomposition.clp.rename(
            amplitude_label="clp_label"
        )
        dataset["matrix"] = optimization_result.fit_decomposition.matrix.rename(
            amplitude_label="clp_label"
        )
        kinetic_elements = [
            element
            for element in optimization_result.elements.values()
            if "compartment" in element.coords
        ]
        if kinetic_elements:
            species_concentration = xr.concat(
                [
                    element["concentrations"].rename(compartment="species")
                    for element in kinetic_elements
                ],
                dim="species",
            )
            species_concentration = species_concentration.isel(
                species=~species_concentration.get_index("species").duplicated()
            )
            concentration_order = [
                dimension
                for dimension in (global_dimension, model_dimension, "species")
                if dimension in species_concentration.dims
            ]
            concentration_order.extend(
                dimension
                for dimension in species_concentration.dims
                if dimension not in concentration_order
            )
            dataset["species_concentration"] = species_concentration.transpose(
                *concentration_order
            )
            species_associated_spectra = xr.concat(
                [element["amplitudes"].rename(compartment="species") for element in kinetic_elements],
                dim="species",
            )
            species_associated_spectra = species_associated_spectra.isel(
                species=~species_associated_spectra.get_index("species").duplicated()
            )
            spectra_order = [
                dimension
                for dimension in (global_dimension, model_dimension, "species")
                if dimension in species_associated_spectra.dims
            ]
            spectra_order.extend(
                dimension
                for dimension in species_associated_spectra.dims
                if dimension not in spectra_order
            )
            dataset["species_associated_spectra"] = species_associated_spectra.transpose(
                *spectra_order
            )
            initial_concentration = xr.concat(
                [
                    element["initial_concentrations"]
                    .isel(activation=0, drop=True)
                    .rename(compartment="species")
                    for element in kinetic_elements
                ],
                dim="species",
            )
            dataset["initial_concentration"] = initial_concentration.isel(
                species=~initial_concentration.get_index("species").duplicated()
            )
        spectral_elements = [
            element
            for element in optimization_result.elements.values()
            if "shape" in element.coords
        ]
        if spectral_elements:
            species_spectra = xr.concat(
                [
                    element["concentrations"].squeeze(drop=True).rename(shape="species")
                    for element in spectral_elements
                ],
                dim="species",
            )
            spectral_order = [
                dimension
                for dimension in (global_dimension, model_dimension, "species")
                if dimension in species_spectra.dims
            ]
            spectral_order.extend(
                dimension for dimension in species_spectra.dims if dimension not in spectral_order
            )
            dataset["species_spectra"] = species_spectra.transpose(*spectral_order)
        dataset.attrs["dataset_scale"] = optimization_result.meta.scale
    return compat_result


def _case_study_matrix_markdown(scheme, element_label, compartments=None):
    """Render a symbolic v0.8 kinetic rate map for legacy notebook display cells."""
    import pandas as pd

    element = scheme.library[element_label]
    compartments = list(compartments or element.compartments)
    table = [["" for _ in compartments] for _ in compartments]
    for (to_compartment, from_compartment), rate in element.rates.items():
        if to_compartment in compartments and from_compartment in compartments:
            table[compartments.index(to_compartment)][compartments.index(from_compartment)] = str(rate)
    return pd.DataFrame(table, index=compartments, columns=compartments).to_markdown()


from cycler import cycler
from glotaran.io import load_parameters
from glotaran.io import save_result
from pyglotaran_extras.inspect import show_a_matrixes
from pyglotaran_extras.plotting.plot_overview import plot_overview
from pyglotaran_extras.plotting.plot_traces import plot_fitted_traces
from pyglotaran_extras.plotting.plot_traces import select_plot_wavelengths

In [ ]:
from pyglotaran_extras import plot_data_overview
DATASETS = {'670TR1': 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_reva.ascii', '670TR2': 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revb.ascii', '700TR1': 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revc.ascii', '700TR2': 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revd.ascii', 'Red1SADS': 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_reve.ascii', 'WLRCSADS': 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revf.ascii', 'Red2SADS': 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revg.ascii', 'WLRP1SADS': 'data/SCy6803WL/synWTred670_700nm_exc2RPnocycle1nm_revh.ascii'}
(fig, axes) = plot_data_overview(DATASETS['670TR1'], nr_of_data_svd_vectors=3, linlog=True, linthresh=0.1)

## Target Analysis

### Used model and parameters

In [ ]:
target_model_path = 'models/20230521model_PSI_TA_SCy6803WL_v08.yml'

In [ ]:
target_parameters_path = 'models/20230521optimized_parameters.csv'
parameters = load_parameters(target_parameters_path)

#### Model file

#### Parameters file

### Create scheme and optimize it

In [ ]:
target_scheme = load_scheme(target_model_path)
target_scheme_parameters = parameters
target_scheme_datasets = DATASETS
target_scheme_dry_run = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=15, dry_run=True, verbose=False, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme load=PASS dry_run=PASS')

In [ ]:
target_result_native = target_scheme.optimize(parameters=target_scheme_parameters, datasets=target_scheme_datasets, maximum_number_function_evaluations=15, raise_exception=True)
print('MIGRATION_VALIDATION scheme=target_scheme real_fit=PASS')
target_result = _case_study_convert(target_result_native, target_scheme)

To save the results of the optimization we can use the `save_result` command.

Because it saves *everything* it consumes about 50MB of disk space per save.

In [ ]:
save_result(result=target_result_native, result_path='results/20230520/result.yaml', allow_overwrite=True)

### Results and parameters

In [ ]:
target_result

In [ ]:
target_result.optimized_parameters

### Amplitude matrices

In [ ]:
show_a_matrixes(target_result)

# Result plots

<sub>Note: The color scheme of the plots in this notebook may not match published figures.</sub>

## Fit quality

In [ ]:
target_result_TA = (target_result.data['670TR1'], target_result.data['670TR2'], target_result.data['700TR1'], target_result.data['700TR2'])
wavelengths = select_plot_wavelengths(target_result_TA, equidistant_wavelengths=True)
plot_fitted_traces(target_result_TA, wavelengths, linlog=True, linthresh=1)

The above command `plot_fitted_traces` is used to plot a selection of traces for a set of wavelengths (autogenerated using the `select_plot_wavelengths` function).
To show to make a manual selection of traces, and 'dress up the plot' see the code below, which reproduces Figure 2 of the paper.

In [ ]:
import warnings
from pyglotaran_extras.plotting.style import ColorCode as cc
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    (fig, ax_) = plot_fitted_traces(target_result_TA, [685, 700, 720, 760], linlog=True, linthresh=1, axes_shape=(2, 2), figsize=(6, 4), title='', per_axis_legend=True, cycler=cycler(color=[cc.grey, cc.black, cc.grey, cc.black, cc.orange, cc.red, cc.orange, cc.red]))
    (handles, labels) = ax_.flatten()[0].get_legend_handles_labels()
    for i in range(len(handles)):
        if i == 1:
            labels[i] = '670 nm excitation'
        elif i == 5:
            labels[i] = '700 nm excitation'
        else:
            labels[i] = '_Hidden'
    for (idx, ax) in enumerate(ax_.flatten()):
        ax.set_ylabel(ax.title.get_text().replace('spectral = ', ''))
        if idx > 1:
            ax.set_xlabel('Time (ps)')
        else:
            ax.set_xlabel('')
        ax.set_title('')
        if ax.get_legend() is not None:
            ax.get_legend().remove()
        for line in ax.lines:
            line.set_linewidth(0.5)
    fig.legend(handles, labels, bbox_to_anchor=(0.5, -0.05), loc='lower center', ncol=len(handles))
    fig.tight_layout()

## Overview 670 exc

In [ ]:
plot_overview(target_result.data['670TR1'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=False, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']))

In [ ]:
plot_overview(target_result.data['670TR2'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=False, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']))

## Overview 700 exc

In [ ]:
plot_overview(target_result.data['700TR1'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=False, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']))

In [ ]:
plot_overview(target_result.data['700TR2'], nr_of_data_svd_vectors=4, nr_of_residual_svd_vectors=1, linlog=False, linthresh=1, cycler=cycler(color=['y', 'g', 'tab:orange', 'r', 'k', 'c', 'b', 'm', 'tab:purple']))

## Comparison of the estimated SADS (orange) and the guidance spectra (blue)
The guidance spectra are (smooth) shapes derived elsewhere 

In [ ]:
target_result.data['Red1SADS'].data.plot()
target_result.data['Red1SADS'].fitted_data.plot()
target_result.data['Red2SADS'].data.plot()
target_result.data['Red2SADS'].fitted_data.plot()

In [ ]:
target_result.data['WLRP1SADS'].data.plot()
target_result.data['WLRP1SADS'].fitted_data.plot()

In [ ]:
target_result.data['WLRCSADS'].data.plot()
target_result.data['WLRCSADS'].fitted_data.plot()